# 🔍 00 — Setup y Verificación del Entorno
**Corre este notebook primero.** Verifica que el pod tiene todo lo necesario antes de entrenar.

Checklist:
- [ ] GPU detectada
- [ ] Datasets en /workspace
- [ ] Dependencias instaladas
- [ ] Módulos del proyecto importan correctamente

## 1. ¿Qué hay en /workspace?

In [1]:
import os

print('=== Contenido de /workspace ===')
for item in sorted(os.listdir('/workspace')):
    full = os.path.join('/workspace', item)
    if os.path.isdir(full):
        try:
            n_files = sum(len(files) for _, _, files in os.walk(full))
            print(f'  📁 {item}/  ({n_files} archivos)')
        except:
            print(f'  📁 {item}/')
    else:
        size_mb = os.path.getsize(full) / 1e6
        print(f'  📄 {item}  ({size_mb:.1f} MB)')

=== Contenido de /workspace ===
  📁 moe_medical_vision/  (152560 archivos)


In [2]:
# Ver uso de disco
!df -h /workspace
print()
!du -sh /workspace/* 2>/dev/null | sort -h

Filesystem                   Size  Used Avail Use% Mounted on
mfs#us-il-1.runpod.net:9421  671T  374T  297T  56% /workspace

108G	/workspace/moe_medical_vision


## 2. GPU disponible

In [3]:
import torch

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        vram_gb = props.total_memory / 1e9
        print(f'  GPU {i}: {props.name} — {vram_gb:.1f} GB VRAM')
else:
    print('  ⚠️  Sin GPU — verifica que el pod está corriendo con GPU')

PyTorch version: 2.4.1+cu124
CUDA disponible: True
  GPU 0: NVIDIA GeForce RTX 4090 — 25.4 GB VRAM


## 3. Instalar dependencias faltantes

In [4]:
!pip install timm faiss-cpu scikit-learn opencv-python nibabel tqdm --quiet

## 4. ¿Dónde están los datasets?

In [7]:
import importlib, sys

# Limpiar cache de imports
for mod in list(sys.modules.keys()):
    if 'datasets' in mod:
        del sys.modules[mod]

sys.path.insert(0, '/workspace/moe_medical_vision/src')

from data.datasets import OsteoarthritisDataset, NIHChestXray14Dataset, ISIC2019Dataset

# Test OA
ds_oa = OsteoarthritisDataset(
    '/workspace/moe_medical_vision/data/raw/osteoporosis/KLGrade/KLGrade',
    split='train'
)
print("OA:", len(ds_oa), ds_oa[0]['image'].shape)

# Test NIH
ds_nih = NIHChestXray14Dataset(
    '/workspace/moe_medical_vision/data/raw/nih', split='train'
)
print("NIH:", len(ds_nih), ds_nih[0]['image'].shape)

# Test ISIC
ds_isic = ISIC2019Dataset(
    '/workspace/moe_medical_vision/data/raw/isic', split='train'
)
print("ISIC:", len(ds_isic), ds_isic[0]['image'].shape)

[OA train] 3813 imgs | Normal:1059 Doubtful:1010 Mild+:1744
OA: 3813 torch.Size([3, 224, 224])
[NIH train] 95302 imágenes
NIH: 95302 torch.Size([3, 224, 224])
[ISIC train] 21532 imágenes
ISIC: 21532 torch.Size([3, 224, 224])


In [6]:
from pathlib import Path

# Buscar los datasets automáticamente en /workspace
POSIBLES_NOMBRES = {
    'nih_chestxray':  ['nih', 'chestxray', 'chest_xray', 'NIH', 'ChestXray'],
    'isic2019':       ['isic', 'ISIC', 'isic2019', 'skin'],
    'osteoarthritis': ['osteo', 'knee', 'arthritis', 'OA'],
    'luna16':         ['luna', 'LUNA', 'lung', 'CT'],
    'pancreatic':     ['pancrea', 'Pancrea', 'abdominal'],
}

workspace = Path('/workspace')
encontrados = {}

for dataset, keywords in POSIBLES_NOMBRES.items():
    for folder in workspace.iterdir():
        if folder.is_dir():
            if any(kw.lower() in folder.name.lower() for kw in keywords):
                encontrados[dataset] = str(folder)
                break

print('Datasets detectados automáticamente:')
for name, path in encontrados.items():
    print(f'  ✅ {name}: {path}')

faltantes = set(POSIBLES_NOMBRES.keys()) - set(encontrados.keys())
if faltantes:
    print(f'\n  ❌ No detectados: {faltantes}')
    print('  → Buscar manualmente con: !find /workspace -name "*.png" | head -5')

# IMPORTANTE: copia estas rutas a config.py
print('\n--- Pega esto en config.py ---')
for name, path in encontrados.items():
    print(f'    "{name}": Path("{path}"),')

Datasets detectados automáticamente:

  ❌ No detectados: {'luna16', 'pancreatic', 'osteoarthritis', 'isic2019', 'nih_chestxray'}
  → Buscar manualmente con: !find /workspace -name "*.png" | head -5

--- Pega esto en config.py ---


In [ ]:
# Si no los detectó, búsqueda manual
print('=== Buscando imágenes médicas en /workspace ===')
!find /workspace -name '*.png' 2>/dev/null | head -10
!find /workspace -name '*.jpg' 2>/dev/null | head -10
!find /workspace -name '*.nii.gz' 2>/dev/null | head -5
!find /workspace -name '*.csv' 2>/dev/null | head -10

## 5. Smoke test de los módulos

In [ ]:
import sys
sys.path.insert(0, '/workspace/moe_project')

import torch
from adaptive_preprocessor import AdaptivePreprocessor, normalize_hu

prep = AdaptivePreprocessor()

# Test 2D
x2d = torch.rand(2, 1, 512, 512)
out2d = prep(x2d)
assert out2d.shape == (2, 3, 224, 224)
print(f'✅ 2D: {x2d.shape} → {out2d.shape}')

# Test 3D
x3d = torch.rand(1, 1, 128, 256, 256)
out3d = prep(x3d)
assert out3d.shape == (1, 1, 64, 64, 64)
print(f'✅ 3D: {x3d.shape} → {out3d.shape}')

# Test en GPU
if torch.cuda.is_available():
    prep_gpu = prep.cuda()
    out_gpu = prep_gpu(x2d.cuda())
    print(f'✅ GPU: {out_gpu.shape} en {out_gpu.device}')

print('\n✅ AdaptivePreprocessor OK')

In [ ]:
from losses import FocalLoss, WeightedBCEWithLogitsLoss, AuxiliaryLoadBalancingLoss

# Test AuxLoss — lo más crítico del proyecto
aux = AuxiliaryLoadBalancingLoss(n_experts=5, alpha=0.01)
gate_probs = torch.softmax(torch.randn(32, 5), dim=-1)
loss, ratio = aux(gate_probs)
print(f'✅ AuxLoss: L={loss.item():.4f}, ratio={ratio:.2f} (límite: 1.30)')
print('✅ Losses OK')

## 6. Test rápido con datos reales (si existen)

In [ ]:
# Cambia esta ruta si el detector automático la encontró arriba
OA_PATH = encontrados.get('osteoarthritis', '/workspace/osteoarthritis')

if Path(OA_PATH).exists():
    from datasets import get_dataloader
    loader, ds = get_dataloader('osteoarthritis', OA_PATH, split='train', batch_size=4, num_workers=0)
    batch = next(iter(loader))
    print(f'✅ Osteoarthritis DataLoader OK')
    print(f'   image shape: {batch["image"].shape}')
    print(f'   label:       {batch["label"]}')
    print(f'   expert_id:   {batch["expert_id"]}')
else:
    print(f'⚠️  Dataset no encontrado en {OA_PATH}')
    print('   Actualiza la ruta manualmente.')

## ✅ Resultado
Si todo verde → ejecutar `01_train_experts_2D.ipynb`  
Si hay errores → revisar rutas en `config.py` y reinstalar deps.